In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

In [2]:
DATA_DIR = Path("/kaggle/input")

for path in DATA_DIR.rglob("*"):
    if path.is_file():
        print(path)

/kaggle/input/datasets/denisebrancaleone/y-sis-hackathon/example_submission.txt
/kaggle/input/datasets/denisebrancaleone/y-sis-hackathon/road_metadata.csv
/kaggle/input/datasets/denisebrancaleone/y-sis-hackathon/train.csv
/kaggle/input/datasets/denisebrancaleone/y-sis-hackathon/test.csv
/kaggle/input/datasets/denisebrancaleone/y-sis-hackathon/incidenti.xlsx
/kaggle/input/datasets/denisebrancaleone/y-sis-hackathon/README.docx


In [3]:
train_path = Path("/kaggle/input/datasets/denisebrancaleone/y-sis-hackathon/train.csv")
test_path = Path("/kaggle/input/datasets/denisebrancaleone/y-sis-hackathon/test.csv")
metadata_path = Path("/kaggle/input/datasets/denisebrancaleone/y-sis-hackathon/road_metadata.csv")
accidents_path = Path("/kaggle/input/datasets/denisebrancaleone/y-sis-hackathon/incidenti.xlsx")

In [6]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
metadata = pd.read_csv(metadata_path)

print("Train:", train.shape)
print("Test:", test.shape)
print("Metadata:", metadata.shape)

display(train.head(10))
train.info()

Train: (292186, 23)
Test: (15048, 2)
Metadata: (627, 1)


,Localita,Provincia,Data Ins,Ora Ins,Canale arrivo,Tipologia,Categoria,Gruppo,Strada,Tipo Persona,...,comuni_attraversati,province_attraversate,km_min,km_max,consistenza_gg_media,leggeri_media,pesanti_media,mese_data,anno,mese
0,NaN,NaN,2016-01-08,16:20:03,E-Mail,Reclami,PEDAGGI AUTOSTRADALI,DG,NaN,CLIENTI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016-01-01,2016,1
1,NaN,NaN,2016-01-11,07:55:12,E-Mail,Informazioni,PEDAGGI AUTOSTRADALI,DG,NaN,CLIENTI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016-01-01,2016,1
2,NaN,RM,2016-01-25,17:09:55,E-Mail,Reclami,INQUINAMENTO ACUSTICO,RM,AU A90 GRANDE RACCORDO ANULARE,CLIENTI,...,1.0,1.0,8.618,66.86,320.125,107702.75,3590.125,2016-01-01,2016,1
3,NaN,NaN,2016-01-27,16:21:00,Telefono,Segnalazioni,MANUTENZIONE STRADALE (RICHIESTE MULTIPLE),DG,NaN,CLIENTI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016-01-01,2016,1
4,BARLETTA,BT,2016-02-02,15:41:56,PEC,Segnalazioni,MANUTENZIONE STRADALE (RICHIESTE MULTIPLE),DG,NaN,ISTITUZIONI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016-02-01,2016,2
5,NaN,NaN,2016-02-05,14:27:49,E-Mail,Sollecito,CLASSIFICAZIONE / COMPETENZA STRADE STATALI,RM,NaN,CLIENTI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016-02-01,2016,2
6,CASTELLAMMARE DI STABIA,NaN,2016-02-09,15:08:35,E-Mail,Informazioni,MANUTENZIONE STRADALE (RICHIESTE MULTIPLE),NaN,NaN,CLIENTI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016-02-01,2016,2
7,BARONISSI,SA,2016-02-10,15:19:58,PEC,Segnalazioni,MANUTENZIONE STRADALE (RICHIESTE MULTIPLE),NaN,NaN,CLIENTI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016-02-01,2016,2
8,TORINO,TO,2016-02-12,18:06:01,Telefono,Sollecito,LAVORI,DG,NaN,CLIENTI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016-02-01,2016,2
9,NaN,NaN,2016-02-15,10:37:04,E-Mail,Segnalazioni,SALA OPERATIVA,NaN,NaN,CLIENTI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016-02-01,2016,2


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 292186 entries, 0 to 292185
Data columns (total 23 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Localita               23817 non-null   object 
 1   Provincia              27174 non-null   object 
 2   Data Ins               292186 non-null  object 
 3   Ora Ins                292186 non-null  object 
 4   Canale arrivo          292185 non-null  object 
 5   Tipologia              292172 non-null  object 
 6   Categoria              292180 non-null  object 
 7   Gruppo                 287924 non-null  object 
 8   Strada                 82274 non-null   object 
 9   Tipo Persona           292186 non-null  object 
 10  file_origine           292186 non-null  int64  
 11  strada_sintetica       78472 non-null   object 
 12  n_tratte               64450 non-null   float64
 13  comuni_attraversati    64450 non-null   float64
 14  province_attraversate  64450 non-nul

In [7]:
assignment_columns = [
    "strada_sintetica",
    "Strada",
    "Provincia",
    "Localita",
    "Gruppo",
    "Data Ins"
]

available_columns = [
    column for column in assignment_columns
    if column in train.columns
]

assignment_audit = pd.DataFrame({
    "non_missing": train[available_columns].notna().sum(),
    "missing": train[available_columns].isna().sum(),
    "missing_percentage": (
        train[available_columns].isna().mean() * 100
    ).round(2),
    "unique_values": train[available_columns].nunique(dropna=True)
})

display(assignment_audit)

,non_missing,missing,missing_percentage,unique_values
strada_sintetica,78472,213714,73.14,329
Strada,82274,209912,71.84,586
Provincia,27174,265012,90.70,112
Localita,23817,268369,91.85,3059
Gruppo,287924,4262,1.46,24
Data Ins,292186,0,0.00,2819


In [8]:
has_synthetic_road = train["strada_sintetica"].notna()
has_original_road = train["Strada"].notna()
has_province = train["Provincia"].notna()
has_locality = train["Localita"].notna()

assignment_groups = pd.Series(
    np.select(
        [
            has_synthetic_road,
            ~has_synthetic_road & has_original_road,
            ~has_synthetic_road & ~has_original_road & has_province,
            ~has_synthetic_road & ~has_original_road & ~has_province & has_locality
        ],
        [
            "A_direct_synthetic_road",
            "B_original_road_only",
            "C_province_only",
            "D_locality_only"
        ],
        default="E_no_geographic_information"
    )
)

display(
    assignment_groups
    .value_counts(dropna=False)
    .rename_axis("assignment_group")
    .reset_index(name="number_of_cases")
)

,assignment_group,number_of_cases
0,E_no_geographic_information,193875
1,A_direct_synthetic_road,78472
2,C_province_only,14792
3,B_original_road_only,3802
4,D_locality_only,1245


In [9]:
road_group = (
    train.loc[
        train["strada_sintetica"].notna()
        & train["Gruppo"].notna(),
        ["strada_sintetica", "Gruppo"]
    ]
    .drop_duplicates()
)

groups_per_road = (
    road_group
    .groupby("strada_sintetica")["Gruppo"]
    .nunique()
    .sort_values(ascending=False)
)

print("Roads analysed:", len(groups_per_road))

print(
    "Roads associated with exactly one Gruppo:",
    (groups_per_road == 1).sum()
)

print(
    "Roads associated with more than one Gruppo:",
    (groups_per_road > 1).sum()
)

display(groups_per_road.value_counts().sort_index())

Roads analysed: 329
Roads associated with exactly one Gruppo: 30
Roads associated with more than one Gruppo: 299


Gruppo
1      30
2      83
3     110
4      61
5      29
6       9
7       3
8       2
9       1
12      1
Name: count, dtype: int64

In [10]:
roads_multiple_groups = groups_per_road[
    groups_per_road > 1
].index

display(
    road_group[
        road_group["strada_sintetica"].isin(roads_multiple_groups)
    ]
    .sort_values(["strada_sintetica", "Gruppo"])
    .head(100)
)

,strada_sintetica,Gruppo
7339,A1,CC
61641,A1,CC_SOCIAL
291811,A1,DG
12842,A18,CC
1322,A18,CT
...,...,...
981,SS12,BO
211396,SS12,CA
962,SS12,CC
27284,SS12,CC_SOCIAL


In [12]:
print(test.shape)
display(test.head(10))

(15048, 2)


,strada,mese_data
0,AU A CATANIA - SIRACUSA,2024-01-01
1,AU A SISTIANA - RABUIESE,2024-01-01
2,AU A18 DIR DIRAMAZIONE DI CATANIA,2024-01-01
3,AU A19 DIR DIRAMAZIONE PER VIA GIAFAR,2024-01-01
4,AU A19 PALERMO - CATANIA,2024-01-01
5,AU A2 AUTOSTRADA DEL MEDITERRANEO,2024-01-01
6,AU A2 DIRAMAZIONE NAPOLI,2024-01-01
7,AU A2 DIRAMAZIONE REGGIO CALABRIA,2024-01-01
8,AU A29 DIR ALCAMO - TRAPANI,2024-01-01
9,AU A29 DIR/A DIRAMAZIONE PER BIRGI,2024-01-01


In [13]:
def normalize_road(series):
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .str.replace('""', '"', regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )

test_roads = set(normalize_road(test["strada"]).dropna())
original_roads = set(normalize_road(train["Strada"]).dropna())
synthetic_roads = set(normalize_road(train["strada_sintetica"]).dropna())

print("Unique roads in test:", len(test_roads))
print()

print("Historical Strada:")
print("Unique values:", len(original_roads))
print("Matches with test:", len(test_roads & original_roads))
print(
    "Percentage of test roads matched:",
    round(100 * len(test_roads & original_roads) / len(test_roads), 2),
    "%"
)
print()

print("Historical strada_sintetica:")
print("Unique values:", len(synthetic_roads))
print("Matches with test:", len(test_roads & synthetic_roads))
print(
    "Percentage of test roads matched:",
    round(100 * len(test_roads & synthetic_roads) / len(test_roads), 2),
    "%"
)

Unique roads in test: 627

Historical Strada:
Unique values: 586
Matches with test: 491
Percentage of test roads matched: 78.31 %

Historical strada_sintetica:
Unique values: 329
Matches with test: 0
Percentage of test roads matched: 0.0 %


In [14]:
unmatched_test = sorted(test_roads - original_roads)
historical_not_in_test = sorted(original_roads - test_roads)

print("Test roads without direct historical match:", len(unmatched_test))
display(
    pd.DataFrame({"unmatched_test_road": unmatched_test}).head(50)
)

print("Historical roads not appearing directly in test:", len(historical_not_in_test))
display(
    pd.DataFrame({
        "historical_road_not_in_test": historical_not_in_test
    }).head(50)
)

Test roads without direct historical match: 136


,unmatched_test_road
0,SS 106 RACC. JONICA
1,SS 116 RANDAZZO CAPO D'ORLANDO
2,SS 117 BIS CENTRALE SICULA
3,SS 117 CENTRALE SICULA
4,SS 118 CORLEONESE AGRIGENTINA
5,SS 119 DI GIBELLINA
6,SS 121 CATANESE
7,SS 122 BIS AGRIGENTINA
8,SS 122 TER TANGENZIALE EST DI CANICATTI'
9,SS 123 DI LICATA


Historical roads not appearing directly in test: 95


,historical_road_not_in_test
0,AU A1-D1 FIANO ROMANO-ROMA NORD
1,AU A1/A11 RACCORDO A1/A11
2,AU A13-D1 RACCORDO FERRARA-PORTO GARIBALDI
3,AU A20 MESSINA-PALERMO
4,AU A24 ROMA - L'AQUILA - TERAMO
5,AU A24 TPU TRONCO DI PENETRAZIONE URBANA DELL'A24
6,AU A25 TORANO - PESCARA
7,AU A30 CASERTA-SALERNO
8,AU A51 TANGENZIALE EST DI MILANO
9,E E78


In [15]:
# Keep only records where both road variables are available
road_family_pairs = (
    train.loc[
        train["Strada"].notna()
        & train["strada_sintetica"].notna(),
        ["Strada", "strada_sintetica"]
    ]
    .drop_duplicates()
)

# Number of synthetic families associated with each detailed road
families_per_road = (
    road_family_pairs
    .groupby("Strada")["strada_sintetica"]
    .nunique()
)

# Number of detailed roads contained in each synthetic family
roads_per_family = (
    road_family_pairs
    .groupby("strada_sintetica")["Strada"]
    .nunique()
    .sort_values(ascending=False)
)

print("Unique detailed Strada analysed:", len(families_per_road))

print(
    "Strada mapped to exactly one strada_sintetica:",
    (families_per_road == 1).sum()
)

print(
    "Strada mapped to more than one strada_sintetica:",
    (families_per_road > 1).sum()
)

print("\nDistribution of synthetic families per Strada:")
display(
    families_per_road
    .value_counts()
    .sort_index()
    .rename_axis("number_of_synthetic_families")
    .reset_index(name="number_of_detailed_roads")
)

print("\nNumber of detailed roads per strada_sintetica:")
display(roads_per_family.describe())

print("\nLargest road families:")
display(
    roads_per_family
    .head(20)
    .rename("number_of_detailed_roads")
    .reset_index()
)

Unique detailed Strada analysed: 517
Strada mapped to exactly one strada_sintetica: 517
Strada mapped to more than one strada_sintetica: 0

Distribution of synthetic families per Strada:


,number_of_synthetic_families,number_of_detailed_roads
0,1,517



Number of detailed roads per strada_sintetica:


count    329.000000
mean       1.571429
std        1.217969
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       12.000000
Name: Strada, dtype: float64


Largest road families:


,strada_sintetica,number_of_detailed_roads
0,SS7,12
1,SS106,9
2,SS14,7
3,SS1,7
4,SS17,6
5,SS12,6
6,A29,5
7,SS16,5
8,SS131,5
9,SS115,5


In [16]:
import re
import pandas as pd

def normalize_family(value):
    """Normalize an existing strada_sintetica value for comparison."""
    if pd.isna(value):
        return pd.NA

    return re.sub(
        r"[^A-Z0-9]",
        "",
        str(value).upper()
    )


def extract_road_family(strada):
    """
    Extract the main road code from the beginning of Strada.

    Examples:
    SS 106 RACC. JONICA       -> SS106
    SS 7 BIS ...              -> SS7
    A29 ...                   -> A29
    AU A1-D1 ...              -> A1
    RA R.A. 11 ...            -> RA11
    NS NSA 167 ...            -> NSA167
    E E78                     -> E78
    """
    if pd.isna(strada):
        return pd.NA

    text = str(strada).upper().strip()
    text = re.sub(r"\s+", " ", text)

    patterns = [
        # Standard state roads
        (r"^SS\s*(\d+)", lambda m: f"SS{m.group(1)}"),

        # Motorways recorded directly
        (r"^A\s*(\d+)", lambda m: f"A{m.group(1)}"),

        # Motorways preceded by AU
        (r"^AU\s+A\s*(\d+)", lambda m: f"A{m.group(1)}"),

        # Raccordi autostradali
        (r"^RA\s+R\.?\s*A\.?\s*(\d+)", lambda m: f"RA{m.group(1)}"),

        # New ANAS roads
        (r"^NS\s+NSA\s*(\d+)", lambda m: f"NSA{m.group(1)}"),

        # European roads
        (r"^E\s+E\s*(\d+)", lambda m: f"E{m.group(1)}"),
    ]

    for pattern, formatter in patterns:
        match = re.search(pattern, text)

        if match:
            return formatter(match)

    return pd.NA


# Use only rows where both identifiers are available
family_validation = (
    train.loc[
        train["Strada"].notna()
        & train["strada_sintetica"].notna(),
        ["Strada", "strada_sintetica"]
    ]
    .drop_duplicates()
    .copy()
)

family_validation["family_extracted"] = (
    family_validation["Strada"]
    .apply(extract_road_family)
)

family_validation["family_known"] = (
    family_validation["strada_sintetica"]
    .apply(normalize_family)
)

family_validation["correct_match"] = (
    family_validation["family_extracted"]
    == family_validation["family_known"]
)

print("Detailed roads evaluated:", len(family_validation))

print(
    "Family successfully extracted:",
    family_validation["family_extracted"].notna().sum()
)

print(
    "Correct matches:",
    family_validation["correct_match"].sum()
)

print(
    "Accuracy:",
    round(
        family_validation["correct_match"].mean() * 100,
        2
    ),
    "%"
)

print("\nMismatches or unresolved cases:")

display(
    family_validation.loc[
        ~family_validation["correct_match"].fillna(False),
        [
            "Strada",
            "strada_sintetica",
            "family_extracted",
            "family_known"
        ]
    ].head(50)
)

Detailed roads evaluated: 517
Family successfully extracted: 517
Correct matches: 517
Accuracy: 100.0 %

Mismatches or unresolved cases:


,Strada,strada_sintetica,family_extracted,family_known


In [17]:
train["road_family"] = train["Strada"].apply(extract_road_family)
test["road_family"] = test["strada"].apply(extract_road_family)

historical_families = set(
    train.loc[train["Strada"].notna(), "road_family"].dropna()
)

test_families = set(test["road_family"].dropna())

unmatched_test_df = (
    test.loc[
        ~normalize_road(test["strada"]).isin(original_roads),
        ["strada", "road_family"]
    ]
    .drop_duplicates()
    .copy()
)

unmatched_test_df["family_has_historical_data"] = (
    unmatched_test_df["road_family"].isin(historical_families)
)

print("Unmatched test roads:", len(unmatched_test_df))

print(
    "Unmatched roads whose family has historical data:",
    unmatched_test_df["family_has_historical_data"].sum()
)

print(
    "Unmatched roads whose family has no historical data:",
    (~unmatched_test_df["family_has_historical_data"]).sum()
)

display(
    unmatched_test_df[
        ~unmatched_test_df["family_has_historical_data"]
    ].head(50)
)

Unmatched test roads: 136
Unmatched roads whose family has historical data: 30
Unmatched roads whose family has no historical data: 106


,strada,road_family,family_has_historical_data
73,SS 116 RANDAZZO CAPO D'ORLANDO,SS116,False
76,SS 118 CORLEONESE AGRIGENTINA,SS118,False
86,SS 121 CATANESE,SS121,False
90,SS 123 DI LICATA,SS123,False
97,SS 127 BIS SETTENTRIONALE SARDA,SS127,False
98,SS 127 SETTENTRIONALE SARDA,SS127,False
101,SS 129 BIS TRASVERSALE SARDA,SS129,False
102,SS 129 TRASVERSALE SARDA,SS129,False
103,SS 13 PONTEBBANA,SS13,False
112,SS 133 BIS DI PALAU,SS133,False


In [19]:
from pathlib import Path
import pandas as pd

metadata_path = list(
    Path("/kaggle/input").rglob("road_metadata.csv")
)[0]

road_metadata = pd.read_csv(
    metadata_path,
    sep=";"
)

print("File:", metadata_path)
print("Shape:", road_metadata.shape)

display(road_metadata.head())

File: /kaggle/input/datasets/denisebrancaleone/y-sis-hackathon/road_metadata.csv
Shape: (627, 9)


,strada,n_tratte,comuni_attraversati,province_attraversate,km_min,km_max,consistenza_gg_media,leggeri_media,pesanti_media
0,AU A CATANIA - SIRACUSA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AU A SISTIANA - RABUIESE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AU A18 DIR DIRAMAZIONE DI CATANIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AU A19 DIR DIRAMAZIONE PER VIA GIAFAR,4.0,4.0,2.0,11.737,92.112,351.25,15007.0,1568.75
4,AU A19 PALERMO - CATANIA,4.0,4.0,2.0,11.737,92.112,351.25,15007.0,1568.75


In [20]:
# Normalize the road identifier in the metadata
road_metadata["road_clean"] = normalize_road(road_metadata["strada"])

# One row per road in the test set
test_road_groups = (
    test[["strada", "road_family"]]
    .drop_duplicates()
    .copy()
)

test_road_groups["road_clean"] = normalize_road(
    test_road_groups["strada"]
)

# Classify the forecasting situation
test_road_groups["forecast_group"] = np.select(
    [
        test_road_groups["road_clean"].isin(original_roads),

        ~test_road_groups["road_clean"].isin(original_roads)
        & test_road_groups["road_family"].isin(historical_families)
    ],
    [
        "seen_road",
        "unseen_road_seen_family"
    ],
    default="unseen_family"
)

# Identify whether metadata exists for each road
metadata_roads = set(
    road_metadata["road_clean"].dropna()
)

test_road_groups["metadata_row_exists"] = (
    test_road_groups["road_clean"].isin(metadata_roads)
)

# Merge the numeric metadata
metadata_columns = [
    "n_tratte",
    "comuni_attraversati",
    "province_attraversate",
    "km_min",
    "km_max",
    "consistenza_gg_media",
    "leggeri_media",
    "pesanti_media"
]

test_road_groups = test_road_groups.merge(
    road_metadata[
        ["road_clean"] + metadata_columns
    ],
    on="road_clean",
    how="left",
    validate="one_to_one"
)

# Check whether at least one numeric metadata value is available
test_road_groups["numeric_metadata_available"] = (
    test_road_groups[metadata_columns]
    .notna()
    .any(axis=1)
)

metadata_by_group = (
    test_road_groups
    .groupby("forecast_group")
    .agg(
        number_of_roads=("strada", "size"),
        metadata_row_exists=("metadata_row_exists", "sum"),
        numeric_metadata_available=(
            "numeric_metadata_available",
            "sum"
        )
    )
)

metadata_by_group["numeric_metadata_missing"] = (
    metadata_by_group["number_of_roads"]
    - metadata_by_group["numeric_metadata_available"]
)

metadata_by_group["percentage_with_numeric_metadata"] = (
    100
    * metadata_by_group["numeric_metadata_available"]
    / metadata_by_group["number_of_roads"]
).round(2)

display(metadata_by_group)

,number_of_roads,metadata_row_exists,numeric_metadata_available,numeric_metadata_missing,percentage_with_numeric_metadata
forecast_group,,,,,
seen_road,491,491,250,241,50.92
unseen_family,106,106,106,0,100.00
unseen_road_seen_family,30,30,30,0,100.00


In [21]:
# Temporary historical month variable
train["data_ins_clean"] = pd.to_datetime(
    train["Data Ins"],
    errors="coerce",
    dayfirst=True
)

train["month_clean"] = (
    train["data_ins_clean"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

train["road_clean"] = normalize_road(train["Strada"])

# Keep only roads that appear exactly in the test set
seen_history = train[
    train["road_clean"].isin(test_roads)
    & train["month_clean"].notna()
].copy()

# Historical coverage by road
road_history_summary = (
    seen_history
    .groupby("road_clean")
    .agg(
        raw_records=("road_clean", "size"),
        observed_months=("month_clean", "nunique"),
        first_observed_month=("month_clean", "min"),
        last_observed_month=("month_clean", "max"),
        observed_years=("data_ins_clean", lambda x: x.dt.year.nunique())
    )
    .reset_index()
)

# Total possible months between 2016-01 and 2023-12
total_historical_months = 8 * 12

road_history_summary["percentage_months_observed"] = (
    100
    * road_history_summary["observed_months"]
    / total_historical_months
).round(2)

print("Seen roads analysed:", len(road_history_summary))

display(
    road_history_summary[
        [
            "raw_records",
            "observed_months",
            "observed_years",
            "percentage_months_observed"
        ]
    ].describe()
)

print("\nRoads with the least historical coverage:")

display(
    road_history_summary
    .sort_values(
        ["observed_months", "raw_records"],
        ascending=True
    )
    .head(30)
)

Seen roads analysed: 452


,raw_records,observed_months,observed_years,percentage_months_observed
count,452.000000,452.000000,452.000000,452.000000
mean,66.108407,22.115044,4.887168,23.036217
std,215.515150,24.674916,2.467188,25.703705
min,1.000000,1.000000,1.000000,1.040000
25%,5.000000,4.000000,3.000000,4.170000
50%,13.000000,11.000000,5.000000,11.460000
75%,45.250000,32.250000,8.000000,33.592500
max,3034.000000,96.000000,8.000000,100.000000



Roads with the least historical coverage:


,road_clean,raw_records,observed_months,first_observed_month,last_observed_month,observed_years,percentage_months_observed
16,NS NSA 215 EX S.S. 51 DI ALEMAGNA,1,1,2022-07-01,2022-07-01,1,1.04
19,NS NSA 371 EX S.S. N. 125 ORIENTALE SARDA,1,1,2019-06-01,2019-06-01,1,1.04
20,NS NSA 571 D'ASPROMONTE,1,1,2023-11-01,2023-11-01,1,1.04
32,RA R.A. 7 RACCORDO AUTOSTRADALE PAVIA - AUTOST...,1,1,2020-06-01,2020-06-01,1,1.04
46,SS 106 DIR./A DI CANTORATO,1,1,2021-03-01,2021-03-01,1,1.04
70,SS 12 VAR. VARIANTE DI ISOLA DELLA SCALA,1,1,2022-05-01,2022-05-01,1,1.04
74,SS 120 RACC DI TRE MONZELLI,1,1,2022-03-01,2022-03-01,1,1.04
89,SS 131 RACC. RACCORDO DI NUORO,1,1,2023-10-01,2023-10-01,1,1.04
95,SS 14 VAR/C VARIANTE DI CAMPALTO,1,1,2023-05-01,2023-05-01,1,1.04
104,SS 16 DIR./C DEL PORTO DI PESCARA,1,1,2021-02-01,2021-02-01,1,1.04


In [23]:
# Roads expected to have exact historical matches
expected_seen_roads = set(
    test_road_groups.loc[
        test_road_groups["forecast_group"] == "seen_road",
        "road_clean"
    ]
)

# Roads that survived the date parsing and coverage calculation
analysed_seen_roads = set(
    road_history_summary["road_clean"]
)

missing_after_date_parsing = sorted(
    expected_seen_roads - analysed_seen_roads
)

print(
    "Matched roads missing from the historical coverage summary:",
    len(missing_after_date_parsing)
)

missing_date_diagnosis = (
    train.loc[
        train["road_clean"].isin(missing_after_date_parsing),
        ["Strada", "Data Ins", "data_ins_clean", "file_origine"]
    ]
    .copy()
)

display(missing_date_diagnosis.head(50))

print("\nRaw date examples:")
display(
    missing_date_diagnosis["Data Ins"]
    .value_counts(dropna=False)
    .head(30)
)

Matched roads missing from the historical coverage summary: 39


,Strada,Data Ins,data_ins_clean,file_origine
391,SS 202 DIR TRIESTINA,2016-05-13,NaT,2016
778,SS 14 RACC. DELLA VENEZIA GIULIA,2016-05-26,NaT,2016
2703,SS 442 DI LACONI E DI URAS,2016-07-18,NaT,2016
4052,SS 344 VAR VARIANTE DI ARCISATE E BISUSCHIO,2016-08-18,NaT,2016
5227,SS 720 VARIANTE DI TAGGIA,2016-09-13,NaT,2016
6595,SS 344 VAR VARIANTE DI ARCISATE E BISUSCHIO,2016-10-18,NaT,2016
6764,SS 442 DI LACONI E DI URAS,2016-10-23,NaT,2016
6777,SS 442 DI LACONI E DI URAS,2016-10-24,NaT,2016
9151,SS 344 VAR VARIANTE DI ARCISATE E BISUSCHIO,2016-12-14,NaT,2016
13858,XS EX 50 BIS VAR EX S.S. 50 BIS VAR DEL GRAPPA...,2017-03-25,NaT,2017



Raw date examples:


Data Ins
2023-07-22    4
2018-10-25    3
2020-06-30    2
2022-04-27    2
2017-09-25    2
2023-11-27    2
2021-05-27    2
2023-06-19    2
2023-09-22    2
2016-05-13    1
2016-07-18    1
2016-08-18    1
2016-09-13    1
2016-05-26    1
2018-01-29    1
2017-11-23    1
2017-06-30    1
2017-03-25    1
2016-10-24    1
2016-12-14    1
2016-10-23    1
2016-10-18    1
2019-11-28    1
2020-01-25    1
2020-01-29    1
2020-07-13    1
2019-07-14    1
2019-08-28    1
2019-02-20    1
2018-10-26    1
Name: count, dtype: int64

In [24]:
train["data_ins_clean"] = pd.to_datetime(
    train["Data Ins"],
    format="%Y-%m-%d",
    errors="coerce"
)

train["month_clean"] = (
    train["data_ins_clean"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

print("Invalid dates:", train["data_ins_clean"].isna().sum())
print("First date:", train["data_ins_clean"].min())
print("Last date:", train["data_ins_clean"].max())

Invalid dates: 0
First date: 2016-01-08 00:00:00
Last date: 2023-12-31 00:00:00


In [25]:
# Recreate the monthly variable using the corrected dates
train["month_clean"] = (
    train["data_ins_clean"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Keep historical records belonging to roads requested in the test set
seen_history = train[
    train["road_clean"].isin(test_roads)
    & train["month_clean"].notna()
].copy()

road_history_summary = (
    seen_history
    .groupby("road_clean")
    .agg(
        raw_records=("road_clean", "size"),
        observed_months=("month_clean", "nunique"),
        first_observed_month=("month_clean", "min"),
        last_observed_month=("month_clean", "max"),
        observed_years=("data_ins_clean", lambda x: x.dt.year.nunique())
    )
    .reset_index()
)

road_history_summary["percentage_months_observed"] = (
    100 * road_history_summary["observed_months"] / 96
).round(2)

print("Seen roads analysed:", len(road_history_summary))

display(
    road_history_summary[
        [
            "raw_records",
            "observed_months",
            "observed_years",
            "percentage_months_observed"
        ]
    ].describe()
)

Seen roads analysed: 491


,raw_records,observed_months,observed_years,percentage_months_observed
count,491.000000,491.000000,491.000000,491.000000
mean,154.749491,23.456212,5.582485,24.433238
std,529.661232,18.956331,2.445074,19.746482
min,1.000000,1.000000,1.000000,1.040000
25%,9.000000,7.000000,3.000000,7.290000
50%,28.000000,18.000000,6.000000,18.750000
75%,110.500000,40.000000,8.000000,41.670000
max,7811.000000,67.000000,8.000000,69.790000


In [26]:
print("Number of columns:", train.shape[1])
print("\nAll columns:")
print(train.columns.tolist())

keywords = [
    "id",
    "case",
    "caso",
    "ticket",
    "richiesta",
    "pratica",
    "protocol",
    "segnalazione",
    "numero",
    "codice"
]

possible_id_columns = [
    column for column in train.columns
    if any(keyword in column.lower() for keyword in keywords)
]

print("\nPossible identifier columns:")
print(possible_id_columns)

if possible_id_columns:
    identifier_audit = pd.DataFrame({
        "data_type": train[possible_id_columns].dtypes.astype(str),
        "non_missing": train[possible_id_columns].notna().sum(),
        "missing": train[possible_id_columns].isna().sum(),
        "unique_values": train[possible_id_columns].nunique(dropna=True)
    })

    identifier_audit["unique_percentage"] = (
        100
        * identifier_audit["unique_values"]
        / identifier_audit["non_missing"]
    ).round(2)

    display(identifier_audit)

Number of columns: 27

All columns:
['Localita', 'Provincia', 'Data Ins', 'Ora Ins', 'Canale arrivo', 'Tipologia', 'Categoria', 'Gruppo', 'Strada', 'Tipo Persona', 'file_origine', 'strada_sintetica', 'n_tratte', 'comuni_attraversati', 'province_attraversate', 'km_min', 'km_max', 'consistenza_gg_media', 'leggeri_media', 'pesanti_media', 'mese_data', 'anno', 'mese', 'road_family', 'data_ins_clean', 'month_clean', 'road_clean']

Possible identifier columns:
[]


In [27]:
original_columns = [
    "Localita",
    "Provincia",
    "Data Ins",
    "Ora Ins",
    "Canale arrivo",
    "Tipologia",
    "Categoria",
    "Gruppo",
    "Strada",
    "Tipo Persona",
    "file_origine",
    "strada_sintetica",
    "n_tratte",
    "comuni_attraversati",
    "province_attraversate",
    "km_min",
    "km_max",
    "consistenza_gg_media",
    "leggeri_media",
    "pesanti_media",
    "mese_data",
    "anno",
    "mese"
]

exact_duplicate_mask = train.duplicated(
    subset=original_columns,
    keep=False
)

print("Rows belonging to an exact duplicate group:",
      exact_duplicate_mask.sum())

print("Exact duplicate rows beyond the first copy:",
      train.duplicated(
          subset=original_columns,
          keep="first"
      ).sum())

print("Percentage duplicated beyond first copy:",
      round(
          100 * train.duplicated(
              subset=original_columns,
              keep="first"
          ).mean(),
          2
      ),
      "%"
)

display(
    train.loc[exact_duplicate_mask, original_columns]
    .sort_values(original_columns)
    .head(30)
)

Rows belonging to an exact duplicate group: 364
Exact duplicate rows beyond the first copy: 195
Percentage duplicated beyond first copy: 0.07 %


,Localita,Provincia,Data Ins,Ora Ins,Canale arrivo,Tipologia,Categoria,Gruppo,Strada,Tipo Persona,...,comuni_attraversati,province_attraversate,km_min,km_max,consistenza_gg_media,leggeri_media,pesanti_media,mese_data,anno,mese
12638,BERGAMO,BG,2017-02-23,19:23:31,E-Mail,Sollecito,ESPROPRI,CC,NaN,CLIENTI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-02-01,2017,2
12639,BERGAMO,BG,2017-02-23,19:23:31,E-Mail,Sollecito,ESPROPRI,CC,NaN,CLIENTI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-02-01,2017,2
158514,CAGLIARI,CA,2021-07-21,14:20:59,PEC,Informazioni,CONCESSIONI AUTORIZZAZIONI E NULLA OSTA,CC,NaN,CLIENTI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-07-01,2021,7
158515,CAGLIARI,CA,2021-07-21,14:20:59,PEC,Informazioni,CONCESSIONI AUTORIZZAZIONI E NULLA OSTA,CC,NaN,CLIENTI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-07-01,2021,7
112235,CALOPEZZATI,CS,2020-02-17,11:03:23,PEC,Informazioni,LAVORI,CZ,SS 106 JONICA,ISTITUZIONI,...,16.0,6.0,11.638,480.503,274.125,11836.500000,651.625000,2020-02-01,2020,2
112236,CALOPEZZATI,CS,2020-02-17,11:03:23,PEC,Informazioni,LAVORI,CZ,SS 106 JONICA,ISTITUZIONI,...,16.0,6.0,11.638,480.503,274.125,11836.500000,651.625000,2020-02-01,2020,2
113825,CALOPEZZATI,CS,2020-03-01,16:33:00,PEC,Informazioni,LAVORI,CZ,SS 106 JONICA,ISTITUZIONI,...,16.0,6.0,11.638,480.503,274.125,11836.500000,651.625000,2020-03-01,2020,3
113826,CALOPEZZATI,CS,2020-03-01,16:33:00,PEC,Informazioni,LAVORI,CZ,SS 106 JONICA,ISTITUZIONI,...,16.0,6.0,11.638,480.503,274.125,11836.500000,651.625000,2020-03-01,2020,3
113831,CALOPEZZATI,CS,2020-03-01,16:38:24,PEC,Informazioni,LAVORI,CZ,SS 106 JONICA,ISTITUZIONI,...,16.0,6.0,11.638,480.503,274.125,11836.500000,651.625000,2020-03-01,2020,3
113832,CALOPEZZATI,CS,2020-03-01,16:38:24,PEC,Informazioni,LAVORI,CZ,SS 106 JONICA,ISTITUZIONI,...,16.0,6.0,11.638,480.503,274.125,11836.500000,651.625000,2020-03-01,2020,3


In [28]:
duplicate_beyond_first = train.duplicated(
    subset=original_columns,
    keep="first"
)

duplicate_impact = train.loc[
    duplicate_beyond_first
].copy()

duplicate_impact["has_road"] = duplicate_impact["Strada"].notna()

duplicate_impact["road_in_test"] = (
    duplicate_impact["road_clean"].isin(test_roads)
)

print("Excess exact duplicates:", len(duplicate_impact))
print(
    "Duplicates with Strada:",
    duplicate_impact["has_road"].sum()
)
print(
    "Duplicates associated with a test road:",
    duplicate_impact["road_in_test"].sum()
)

display(
    duplicate_impact[
        duplicate_impact["road_in_test"]
    ][
        ["Strada", "Data Ins", "Ora Ins", "Tipologia", "Categoria"]
    ].head(30)
)

Excess exact duplicates: 195
Duplicates with Strada: 54
Duplicates associated with a test road: 52


,Strada,Data Ins,Ora Ins,Tipologia,Categoria
2315,SS 80 DEL GRAN SASSO D'ITALIA,2016-07-08,13:31:53,Legge 241/90,CONCESSIONI AUTORIZZAZIONI E NULLA OSTA
4519,AU A90 GRANDE RACCORDO ANULARE,2016-08-30,09:18:52,Segnalazioni,BARRIERE DI SICUREZZA
7273,SS 36 DEL LAGO DI COMO E DELLO SPLUGA,2016-11-04,15:55:53,Reclami,MANUTENZIONE STRADALE (RICHIESTE MULTIPLE)
9633,SS 38 DELLO STELVIO,2016-12-27,09:07:03,Informazioni,FATTURE
19531,SS 585 FONDO VALLE DEL NOCE,2017-07-25,14:43:32,Sala Operativa Nazionale,INQUINAMENTO IDRICO (MATERIALE INQUINANTE)
23311,SS 62 DELLA CISA,2017-10-13,19:32:19,Sollecito,INFORMAZIONI VARIE
23313,SS 62 DELLA CISA,2017-10-13,19:32:40,Sollecito,INFORMAZIONI VARIE
28975,SS 309 ROMEA,2018-01-30,09:42:02,Reclami,VIABILITA'
28981,SS 145 SORRENTINA,2018-01-30,09:42:05,Sala Operativa Nazionale,OGGETTO O LIQUIDO SU STRADA
35490,E E45,2018-04-30,11:29:38,Segnalazioni,LAVORI


In [29]:
case_signature_columns = [
    "Localita",
    "Provincia",
    "Data Ins",
    "Ora Ins",
    "Canale arrivo",
    "Tipologia",
    "Categoria",
    "Gruppo",
    "Strada",
    "Tipo Persona",
    "strada_sintetica"
]

signature_summary = (
    train
    .groupby(
        case_signature_columns,
        dropna=False
    )
    .agg(
        number_of_rows=("file_origine", "size"),
        number_of_source_files=("file_origine", "nunique"),
        first_source_file=("file_origine", "min"),
        last_source_file=("file_origine", "max")
    )
    .reset_index()
)

repeated_across_sources = signature_summary[
    signature_summary["number_of_source_files"] > 1
].copy()

print(
    "Candidate cases appearing in multiple source files:",
    len(repeated_across_sources)
)

print(
    "Total rows represented by these candidates:",
    repeated_across_sources["number_of_rows"].sum()
)

print(
    "Potential excess rows beyond one per candidate case:",
    (
        repeated_across_sources["number_of_rows"] - 1
    ).sum()
)

display(
    repeated_across_sources
    .sort_values(
        ["number_of_source_files", "number_of_rows"],
        ascending=False
    )
    .head(30)
)

Candidate cases appearing in multiple source files: 0
Total rows represented by these candidates: 0
Potential excess rows beyond one per candidate case: 0


,Localita,Provincia,Data Ins,Ora Ins,Canale arrivo,Tipologia,Categoria,Gruppo,Strada,Tipo Persona,strada_sintetica,number_of_rows,number_of_source_files,first_source_file,last_source_file


In [30]:
train["registration_datetime"] = pd.to_datetime(
    train["Data Ins"].astype(str)
    + " "
    + train["Ora Ins"].astype(str),
    errors="coerce"
)

timestamp_summary = (
    train.groupby("registration_datetime", dropna=False)
    .agg(
        number_of_rows=("registration_datetime", "size"),
        number_of_roads=("Strada", "nunique"),
        number_of_categories=("Categoria", "nunique"),
        number_of_groups=("Gruppo", "nunique")
    )
    .reset_index()
)

print("Valid registration timestamps:",
      train["registration_datetime"].notna().sum())

print("Unique registration timestamps:",
      train["registration_datetime"].nunique())

print(
    "Timestamps appearing more than once:",
    (timestamp_summary["number_of_rows"] > 1).sum()
)

print(
    "Rows belonging to repeated timestamps:",
    timestamp_summary.loc[
        timestamp_summary["number_of_rows"] > 1,
        "number_of_rows"
    ].sum()
)

display(
    timestamp_summary[
        timestamp_summary["number_of_rows"] > 1
    ]
    .sort_values("number_of_rows", ascending=False)
    .head(30)
)

Valid registration timestamps: 223855
Unique registration timestamps: 222512
Timestamps appearing more than once: 1283
Rows belonging to repeated timestamps: 70956


,registration_datetime,number_of_rows,number_of_roads,number_of_categories,number_of_groups
222512,NaT,68331,464,75,22
36566,2018-03-08 06:24:48,7,1,1,1
36564,2018-03-08 06:24:45,5,1,1,1
28167,2017-12-20 10:03:55,4,3,4,3
215466,2022-10-28 18:59:16,4,0,1,1
215467,2022-10-28 18:59:17,3,0,1,1
205660,2022-07-29 11:53:34,3,0,1,1
182112,2022-01-18 09:59:48,3,2,3,2
215468,2022-10-28 18:59:18,3,0,1,1
113437,2020-03-01 16:38:24,3,1,1,1


In [31]:
stable_case_columns = [
    "registration_datetime",
    "Strada",
    "Localita",
    "Provincia",
    "Canale arrivo",
    "Tipo Persona"
]

candidate_case_groups = (
    train.loc[
        train["registration_datetime"].notna()
    ]
    .groupby(
        stable_case_columns,
        dropna=False
    )
    .agg(
        number_of_rows=("registration_datetime", "size"),
        number_of_tipologie=("Tipologia", "nunique"),
        number_of_categorie=("Categoria", "nunique"),
        number_of_gruppi=("Gruppo", "nunique"),
        number_of_source_files=("file_origine", "nunique")
    )
    .reset_index()
)

repeated_candidate_cases = candidate_case_groups[
    candidate_case_groups["number_of_rows"] > 1
].copy()

print(
    "Candidate repeated cases:",
    len(repeated_candidate_cases)
)

print(
    "Rows represented:",
    repeated_candidate_cases["number_of_rows"].sum()
)

print(
    "Potential excess rows:",
    (
        repeated_candidate_cases["number_of_rows"] - 1
    ).sum()
)

display(
    repeated_candidate_cases
    .sort_values("number_of_rows", ascending=False)
    .head(30)
)

Candidate repeated cases: 307
Rows represented: 646
Potential excess rows: 339


,registration_datetime,Strada,Localita,Provincia,Canale arrivo,Tipo Persona,number_of_rows,number_of_tipologie,number_of_categorie,number_of_gruppi,number_of_source_files
36671,2018-03-08 06:24:48,SS 64 PORRETTANA,NaN,BO,Sito Internet,CLIENTI,7,1,1,1,1
36669,2018-03-08 06:24:45,SS 64 PORRETTANA,NaN,BO,Sito Internet,CLIENTI,5,1,1,1,1
216419,2022-10-28 18:59:16,NaN,NaN,NaN,E-Mail,CLIENTI,4,1,1,1,1
216440,2022-10-28 19:36:57,NaN,NaN,NaN,E-Mail,CLIENTI,3,1,2,1,1
172955,2021-11-02 12:09:15,NaN,NaN,NaN,E-Mail,CLIENTI,3,1,2,1,1
113710,2020-03-01 16:38:24,SS 106 JONICA,CALOPEZZATI,CS,PEC,ISTITUZIONI,3,1,1,1,1
22068,2017-09-19 12:02:33,NaN,NaN,CR,Sito Internet,CLIENTI,3,1,1,1,1
216430,2022-10-28 19:12:58,NaN,NaN,NaN,E-Mail,CLIENTI,3,1,1,1,1
206571,2022-07-29 11:53:37,NaN,NaN,NaN,E-Mail,CLIENTI,3,1,1,1,1
206573,2022-07-29 11:53:39,NaN,NaN,NaN,E-Mail,CLIENTI,3,1,1,1,1


In [32]:
import numpy as np
import pandas as pd
import re

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_log_error


# ---------------------------------------------------------
# 1. Helper functions
# ---------------------------------------------------------

def normalize_road(series):
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .str.replace('""', '"', regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )


def extract_road_family(strada):
    if pd.isna(strada):
        return pd.NA

    text = str(strada).upper().strip()
    text = re.sub(r"\s+", " ", text)

    patterns = [
        (r"^SS\s*(\d+)", lambda m: f"SS{m.group(1)}"),
        (r"^A\s*(\d+)", lambda m: f"A{m.group(1)}"),
        (r"^AU\s+A\s*(\d+)", lambda m: f"A{m.group(1)}"),
        (r"^RA\s+R\.?\s*A\.?\s*(\d+)", lambda m: f"RA{m.group(1)}"),
        (r"^NS\s+NSA\s*(\d+)", lambda m: f"NSA{m.group(1)}"),
        (r"^E\s+E\s*(\d+)", lambda m: f"E{m.group(1)}"),
    ]

    for pattern, formatter in patterns:
        match = re.search(pattern, text)
        if match:
            return formatter(match)

    # Fallback: first token
    return text.split()[0] if text else pd.NA


# ---------------------------------------------------------
# 2. Correct dates and road identifiers
# ---------------------------------------------------------

train["data_ins_clean"] = pd.to_datetime(
    train["Data Ins"],
    format="%Y-%m-%d",
    errors="coerce"
)

train["month_clean"] = (
    train["data_ins_clean"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

train["road_clean"] = normalize_road(train["Strada"])
train["road_family"] = train["Strada"].apply(extract_road_family)

test["road_clean"] = normalize_road(test["strada"])
test["road_family"] = test["strada"].apply(extract_road_family)
test["month_clean"] = pd.to_datetime(test["mese_data"])


# ---------------------------------------------------------
# 3. Remove only exact duplicates
# ---------------------------------------------------------

original_columns = [
    "Localita",
    "Provincia",
    "Data Ins",
    "Ora Ins",
    "Canale arrivo",
    "Tipologia",
    "Categoria",
    "Gruppo",
    "Strada",
    "Tipo Persona",
    "file_origine",
    "strada_sintetica",
    "n_tratte",
    "comuni_attraversati",
    "province_attraversate",
    "km_min",
    "km_max",
    "consistenza_gg_media",
    "leggeri_media",
    "pesanti_media",
    "mese_data",
    "anno",
    "mese"
]

train_deduplicated = train.drop_duplicates(
    subset=original_columns,
    keep="first"
).copy()

print("Original rows:", len(train))
print("Rows after exact deduplication:", len(train_deduplicated))


# ---------------------------------------------------------
# 4. Create observed road-month target
# ---------------------------------------------------------

monthly_target = (
    train_deduplicated.loc[
        train_deduplicated["road_clean"].notna()
        & train_deduplicated["month_clean"].notna()
    ]
    .groupby(
        ["road_clean", "road_family", "month_clean"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "y"})
)

monthly_target["year"] = monthly_target["month_clean"].dt.year
monthly_target["month"] = monthly_target["month_clean"].dt.month

print("Observed road-month targets:", len(monthly_target))
display(monthly_target.head())
display(monthly_target["y"].describe())

Original rows: 292186
Rows after exact deduplication: 291991
Observed road-month targets: 11908


,road_clean,road_family,month_clean,y,year,month
0,AU A CATANIA - SIRACUSA,AU,2016-06-01,3,2016,6
1,AU A CATANIA - SIRACUSA,AU,2016-08-01,4,2016,8
2,AU A CATANIA - SIRACUSA,AU,2016-09-01,10,2016,9
3,AU A CATANIA - SIRACUSA,AU,2017-03-01,1,2017,3
4,AU A CATANIA - SIRACUSA,AU,2017-04-01,6,2017,4


count    11908.000000
mean         6.904602
std         19.468614
min          1.000000
25%          1.000000
50%          2.000000
75%          6.000000
max        466.000000
Name: y, dtype: float64

In [33]:
# Normalize metadata road identifier
road_metadata["road_clean"] = normalize_road(
    road_metadata["strada"]
)

metadata_columns = [
    "n_tratte",
    "comuni_attraversati",
    "province_attraversate",
    "km_min",
    "km_max",
    "consistenza_gg_media",
    "leggeri_media",
    "pesanti_media"
]

# Conservative numeric conversion:
# malformed values become missing rather than being guessed
for column in metadata_columns:
    road_metadata[column] = pd.to_numeric(
        road_metadata[column],
        errors="coerce"
    )

road_metadata["road_length_km"] = (
    road_metadata["km_max"] -
    road_metadata["km_min"]
)

road_metadata["traffic_total"] = (
    road_metadata["leggeri_media"] +
    road_metadata["pesanti_media"]
)

road_metadata["heavy_vehicle_share"] = (
    road_metadata["pesanti_media"] /
    road_metadata["traffic_total"].replace(0, np.nan)
)

road_metadata["metadata_available"] = (
    road_metadata[metadata_columns]
    .notna()
    .any(axis=1)
    .astype(int)
)

model_metadata_columns = metadata_columns + [
    "road_length_km",
    "traffic_total",
    "heavy_vehicle_share",
    "metadata_available"
]

road_metadata_model = (
    road_metadata[
        ["road_clean"] + model_metadata_columns
    ]
    .drop_duplicates("road_clean")
)

print(road_metadata_model.shape)
display(road_metadata_model.head())

(627, 13)


,road_clean,n_tratte,comuni_attraversati,province_attraversate,km_min,km_max,consistenza_gg_media,leggeri_media,pesanti_media,road_length_km,traffic_total,heavy_vehicle_share,metadata_available
0,AU A CATANIA - SIRACUSA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,AU A SISTIANA - RABUIESE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,AU A18 DIR DIRAMAZIONE DI CATANIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,AU A19 DIR DIRAMAZIONE PER VIA GIAFAR,4.0,4.0,2.0,11.737,92.112,351.25,15007.0,1568.75,80.375,16575.75,0.094641,1
4,AU A19 PALERMO - CATANIA,4.0,4.0,2.0,11.737,92.112,351.25,15007.0,1568.75,80.375,16575.75,0.094641,1


In [34]:
def add_historical_features(data, reference):
    """
    Historical target aggregates are calculated only from reference data.

    For validation:
        reference = 2016-2022

    For final test:
        reference = 2016-2023
    """

    result = data.copy()

    road_stats = (
        reference.groupby("road_clean")["y"]
        .agg(
            road_mean="mean",
            road_median="median",
            road_max="max",
            road_observed_months="count"
        )
        .reset_index()
    )

    family_stats = (
        reference.groupby("road_family")["y"]
        .agg(
            family_mean="mean",
            family_median="median",
            family_max="max",
            family_observed_months="count"
        )
        .reset_index()
    )

    calendar_month_stats = (
        reference.groupby("month")["y"]
        .agg(
            calendar_month_mean="mean",
            calendar_month_median="median"
        )
        .reset_index()
    )

    result = result.merge(
        road_stats,
        on="road_clean",
        how="left"
    )

    result = result.merge(
        family_stats,
        on="road_family",
        how="left"
    )

    result = result.merge(
        calendar_month_stats,
        on="month",
        how="left"
    )

    result["global_mean"] = reference["y"].mean()
    result["global_median"] = reference["y"].median()

    result["road_has_history"] = (
        result["road_observed_months"].notna().astype(int)
    )

    result["family_has_history"] = (
        result["family_observed_months"].notna().astype(int)
    )

    return result

In [35]:
# Temporal split
train_fold = monthly_target[
    monthly_target["year"] <= 2022
].copy()

validation_fold = monthly_target[
    monthly_target["year"] == 2023
].copy()

# Build features without using 2023 targets
train_fold_features = add_historical_features(
    train_fold,
    train_fold
)

validation_fold_features = add_historical_features(
    validation_fold,
    train_fold
)

# Merge road metadata
train_fold_features = train_fold_features.merge(
    road_metadata_model,
    on="road_clean",
    how="left"
)

validation_fold_features = validation_fold_features.merge(
    road_metadata_model,
    on="road_clean",
    how="left"
)

# Road type
train_fold_features["road_type"] = (
    train_fold_features["road_clean"]
    .str.extract(r"^([A-Z]+)", expand=False)
    .fillna("UNKNOWN")
)

validation_fold_features["road_type"] = (
    validation_fold_features["road_clean"]
    .str.extract(r"^([A-Z]+)", expand=False)
    .fillna("UNKNOWN")
)

# Time index
train_fold_features["time_index"] = (
    (train_fold_features["year"] - 2016) * 12
    + train_fold_features["month"]
)

validation_fold_features["time_index"] = (
    (validation_fold_features["year"] - 2016) * 12
    + validation_fold_features["month"]
)

In [36]:
categorical_features = [
    "road_clean",
    "road_family",
    "road_type",
    "month"
]

numeric_features = [
    "year",
    "time_index",
    "road_mean",
    "road_median",
    "road_max",
    "road_observed_months",
    "family_mean",
    "family_median",
    "family_max",
    "family_observed_months",
    "calendar_month_mean",
    "calendar_month_median",
    "global_mean",
    "global_median",
    "road_has_history",
    "family_has_history"
] + model_metadata_columns

feature_columns = categorical_features + numeric_features

X_train = train_fold_features[feature_columns].copy()
y_train = train_fold_features["y"].copy()

X_valid = validation_fold_features[feature_columns].copy()
y_valid = validation_fold_features["y"].copy()

# CatBoost categorical columns cannot contain nulls
for column in categorical_features:
    X_train[column] = X_train[column].astype("string").fillna("MISSING")
    X_valid[column] = X_valid[column].astype("string").fillna("MISSING")

In [37]:
catboost_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    l2_leaf_reg=5,
    verbose=100,
    allow_writing_files=False
)

catboost_model.fit(
    X_train,
    np.log1p(y_train),
    cat_features=categorical_features,
    eval_set=(X_valid, np.log1p(y_valid)),
    early_stopping_rounds=100,
    verbose=100
)

validation_prediction_log = catboost_model.predict(X_valid)

validation_prediction = np.clip(
    np.expm1(validation_prediction_log),
    0,
    None
)

validation_rmsle = np.sqrt(
    mean_squared_log_error(
        y_valid,
        validation_prediction
    )
)

print("2023 validation RMSLE:", validation_rmsle)

0:	learn: 0.8219491	test: 0.9297840	best: 0.9297840 (0)	total: 67.5ms	remaining: 1m 7s
100:	learn: 0.3923207	test: 0.6325298	best: 0.6314785 (70)	total: 1.38s	remaining: 12.3s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6314785373
bestIteration = 70

Shrink model to first 71 iterations.
2023 validation RMSLE: 0.6314785320466426


In [38]:
# Prepare complete training data
full_train_features = add_historical_features(
    monthly_target,
    monthly_target
)

full_train_features = full_train_features.merge(
    road_metadata_model,
    on="road_clean",
    how="left"
)

full_train_features["road_type"] = (
    full_train_features["road_clean"]
    .str.extract(r"^([A-Z]+)", expand=False)
    .fillna("UNKNOWN")
)

full_train_features["time_index"] = (
    (full_train_features["year"] - 2016) * 12
    + full_train_features["month"]
)


# Prepare test rows in their original submission order
test_features = test.copy()

test_features["year"] = test_features["month_clean"].dt.year
test_features["month"] = test_features["month_clean"].dt.month

test_features = add_historical_features(
    test_features,
    monthly_target
)

test_features = test_features.merge(
    road_metadata_model,
    on="road_clean",
    how="left"
)

test_features["road_type"] = (
    test_features["road_clean"]
    .str.extract(r"^([A-Z]+)", expand=False)
    .fillna("UNKNOWN")
)

test_features["time_index"] = (
    (test_features["year"] - 2016) * 12
    + test_features["month"]
)


X_full = full_train_features[feature_columns].copy()
y_full = full_train_features["y"].copy()

X_test = test_features[feature_columns].copy()

for column in categorical_features:
    X_full[column] = X_full[column].astype("string").fillna("MISSING")
    X_test[column] = X_test[column].astype("string").fillna("MISSING")

In [39]:
final_catboost_model = CatBoostRegressor(
    iterations=catboost_model.get_best_iteration() + 1,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    l2_leaf_reg=5,
    verbose=100,
    allow_writing_files=False
)

final_catboost_model.fit(
    X_full,
    np.log1p(y_full),
    cat_features=categorical_features,
    verbose=100
)

test_prediction = np.clip(
    np.expm1(
        final_catboost_model.predict(X_test)
    ),
    0,
    None
)

print("Predictions:", len(test_prediction))
print("Minimum:", test_prediction.min())
print("Maximum:", test_prediction.max())
print("Mean:", test_prediction.mean())

assert len(test_prediction) == len(test)
assert len(test_prediction) == 15048

0:	learn: 0.8415815	total: 21.6ms	remaining: 1.51s
70:	learn: 0.4061177	total: 642ms	remaining: 0us
Predictions: 15048
Minimum: 1.143664845816047
Maximum: 132.29302295381112
Mean: 3.110284018391405


In [40]:
submission = pd.DataFrame({
    "prediction": test_prediction
})

submission.to_csv(
    "/kaggle/working/submission_catboost.csv",
    index=False,
    header=False
)

display(submission.head(20))

,prediction
0,4.113962
1,1.520286
2,1.823825
3,1.271410
4,20.805955
5,115.103886
6,1.455291
7,1.983868
8,2.383277
9,1.336030


In [41]:
# Convert predictions to non-negative integer counts
test_prediction_integer = np.rint(
    np.clip(test_prediction, 0, None)
).astype(int)

submission = pd.DataFrame({
    "prediction": test_prediction_integer
})

submission.to_csv(
    "/kaggle/working/submission_catboost.csv",
    index=False,
    header=False
)

print("Number of predictions:", len(submission))
print("Minimum prediction:", submission["prediction"].min())
print("Maximum prediction:", submission["prediction"].max())

display(submission.head(20))

Number of predictions: 15048
Minimum prediction: 1
Maximum prediction: 132


,prediction
0,4
1,2
2,2
3,1
4,21
5,115
6,1
7,2
8,2
9,1


In [42]:
# RMSLE with continuous predictions
rmsle_continuous = np.sqrt(
    mean_squared_log_error(
        y_valid,
        validation_prediction
    )
)

# RMSLE after integer rounding
validation_prediction_integer = np.rint(
    np.clip(validation_prediction, 0, None)
).astype(int)

rmsle_integer = np.sqrt(
    mean_squared_log_error(
        y_valid,
        validation_prediction_integer
    )
)

print("Continuous RMSLE:", rmsle_continuous)
print("Integer RMSLE:", rmsle_integer)

Continuous RMSLE: 0.6314785320466426
Integer RMSLE: 0.6448168260898833


In [44]:
import pandas as pd

submission_check = pd.read_csv(
    "/kaggle/working/submission_catboost.csv",
    header=None
)

print(submission_check.shape)
display(submission_check.head())

(15048, 1)


,0
0,4
1,2
2,2
3,1
4,21


In [46]:
import os

file_path = "/kaggle/working/submission_catboost.csv"

print("File exists:", os.path.exists(file_path))
print("File size:", os.path.getsize(file_path) if os.path.exists(file_path) else "Not found")

File exists: True
File size: 30930


In [48]:
import os
from IPython.display import FileLink, display

os.chdir("/kaggle/working")
display(FileLink("submission_catboost.csv"))

/kaggle/working/submission_catboost.csv